<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2026/W11_strawberry_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Strawberry Challenge**

How many r's are in "strawberry"? LLMs write fluent text, yet they often fail at a task that a child can solve: counting how often a letter appears in a word. In this challenge, you write prompts for `Llama-3.2-3B-Instruct`, the instruction-tuned model from Part 2 of Exercise 5, and try to make it count letters correctly. Use what you know about zero-shot, few-shot and chain-of-thought prompting.

How it works:

1. Load the model and look at why the task is hard for it.
2. Evaluate a simple zero-shot prompt on 20 practice words.
3. Write your own prompt and improve it.
4. Submit your prompt. The submissions are evaluated on 30 hidden words, and a leaderboard shows the best prompt of each team.

You need a GPU (in Colab: "Runtime" → "Change runtime type" → T4 GPU). Run the first cells right away: the model is downloaded the first time you load it, which takes a few minutes.

In [ ]:
%pip install -q transformers==5.16.1

import re
import time
import torch
from transformers import pipeline
from transformers.utils import logging

logging.set_verbosity_error() # hide warnings from transformers that do not matter here

# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
# Half precision on a GPU, full precision on a CPU
dtype = torch.float16 if device != 'cpu' else torch.float32
print(device)

## 1) Loading the model

We load the model with a `text-generation` pipeline. It comes from the `unsloth` organisation on the Hugging Face Hub, which hosts copies of Meta's Llama 3.2 models that you can download without requesting access first (the Llama 3.2 licence still applies). The weights take about 6.4 GB.

In [ ]:
pipe = pipeline("text-generation", model="unsloth/Llama-3.2-3B-Instruct", device=device, dtype=dtype)
pipe.tokenizer.padding_side = "left" # we generate for several prompts at once

## 2) Why is this task hard for an LLM?

The model does not see letters. The tokenizer splits the text into tokens, most of them pieces of words, and the model only sees the ID of each token. Look at how some of the words are split.

In [ ]:
for word in ["strawberry", "mississippi", "bookkeeper", "hippopotamus"]:
    print(f"{word:13} → {pipe.tokenizer.tokenize(word)}")

To count the r's in "strawberry", the model has to know how each of its tokens is spelled. Nobody tells it that during training; it can only pick it up indirectly from the texts it has seen. Keep this in mind when you design your prompts.

## 3) The practice words and the rules

Each practice item is a word with a target letter. We compute the correct answer with Python's `str.count`.

In [ ]:
practice = [
    ("strawberry", "r"), ("banana", "a"), ("parallel", "l"), ("cappuccino", "c"),
    ("pepper", "p"), ("committee", "t"), ("necessary", "s"), ("tomorrow", "r"),
    ("embarrass", "r"), ("rhythm", "h"), ("mississippi", "s"), ("referee", "e"),
    ("independence", "e"), ("senseless", "s"), ("abracadabra", "a"), ("chocolate", "l"),
    ("coffee", "o"), ("hippopotamus", "m"), ("python", "e"), ("bookkeeper", "a"),
]

for word, letter in practice[:5]:
    print(f"{word}, {letter}: {word.count(letter)}")

**Rules**

- Everyone uses the same model and the same generation settings: greedy decoding and at most 384 new tokens. They are fixed in `evaluate` below.
- A prompt consists of a system message, which may be empty, and a template for the user message. Write `{word}` and `{letter}` in the template where the word and the letter should go. Apart from that, you can write anything into the two messages, including examples.
- The model's answer is the last number written in digits in its output. Words such as "three" do not count.
- Examples are welcome, but do not use the practice words in them; otherwise, your practice score tells you nothing.
- Each team can submit at most 5 prompts, and its best one counts. The best accuracy on the hidden words wins. If two prompts are equally good, the shorter one (in tokens) wins.

In [ ]:
def extract_answer(text):
    """Returns the last number written in digits, or None if there is none."""
    numbers = re.findall(r"\d+", text)
    return int(numbers[-1]) if numbers else None


def build_messages(system_prompt, user_template, word, letter):
    """Fills the template and returns the list of messages for the chat template."""
    user_message = user_template.replace("{word}", word).replace("{letter}", letter)
    messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
    return messages + [{"role": "user", "content": user_message}]


def generate(system_prompt, user_template, pairs, model):
    """Generates greedily for all (word, letter) pairs and returns the outputs."""
    messages = [build_messages(system_prompt, user_template, word, letter) for word, letter in pairs]
    outputs = model(messages, max_new_tokens=384, do_sample=False, return_full_text=False, batch_size=len(messages))
    return [output[0]["generated_text"] for output in outputs]


def evaluate(system_prompt, user_template, pairs=practice, model=pipe, show_errors=3):
    """Prints the accuracy of a prompt on the pairs and a few wrong outputs."""
    start = time.time()
    outputs = generate(system_prompt, user_template, pairs, model)
    results = [(word, letter, word.count(letter), extract_answer(output), output)
               for (word, letter), output in zip(pairs, outputs)]
    errors = [result for result in results if result[2] != result[3]]
    accuracy = (len(results) - len(errors)) / len(results)
    print(f"Accuracy: {accuracy:.2f} ({len(results) - len(errors)}/{len(results)}), {time.time() - start:.0f} s")
    for word, letter, correct, answer, output in errors[:show_errors]:
        print(f"\n--- {word}, {letter}: correct {correct}, answer {answer}\n{output}")

## 4) Baseline: a zero-shot prompt

In [ ]:
system_prompt = ""
user_template = "How many times does the letter '{letter}' appear in the word '{word}'? Answer with a number."

evaluate(system_prompt, user_template)

Look at the wrong answers. Is the model far off or close? Does it give a number at all?

## 5) Your turn

Write your own prompt and evaluate it. Some ideas:

- **Few-shot:** show the model a few solved examples with other words.
- **Chain-of-thought:** let the model write out intermediate steps before the final number. Which steps would help with this task?
- **Format:** make sure that the final number is the last number in the output.
- **System message:** give the model a role or rules to follow.

Keep track of what you try and how well it works. Every evaluation takes a few seconds, so you can try many versions. A triple-quoted string (`"""..."""`) lets you write a prompt over several lines.

In [ ]:
system_prompt = """"""

user_template = """How many times does the letter '{letter}' appear in the word '{word}'? Answer with a number."""

evaluate(system_prompt, user_template)

To see the whole output for one word, use `ask`:

In [ ]:
def ask(system_prompt, user_template, word, letter, model=pipe):
    """Prints the full output for one word, the extracted answer and the correct answer."""
    output = generate(system_prompt, user_template, [(word, letter)], model)[0]
    print(output)
    print(f"\nAnswer: {extract_answer(output)} | correct: {word.count(letter)}")

ask(system_prompt, user_template, "strawberry", "r")

## 6) Submitting your prompt

Open the address shown in class in your browser. There, you enter your team name and copy your system message and user template into the form. The server evaluates the prompt on the 30 hidden words with the same model and settings as `evaluate` and shows its accuracy and the leaderboard. Each team has at most 5 attempts, so use the same team name every time.

The next cell prints your current prompt so that you can copy it.

In [ ]:
length = len(pipe.tokenizer(system_prompt + user_template, add_special_tokens=False)["input_ids"])
print(f"System message:\n{system_prompt}\n")
print(f"User template:\n{user_template}\n")
print(f"Length: {length} tokens")